# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mhv-codes/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [3]:

from google.colab import userdata
from huggingface_hub import HfApi

token = userdata.get("HF_TOKEN")

api = HfApi(token=token)
dataset = api.dataset_info("FlyRank/internship-warehouse")

print("✅ Connected to FlyRank dataset!")
print("Dataset:", dataset.id)


✅ Connected to FlyRank dataset!
Dataset: FlyRank/internship-warehouse


My unit of analysis and time window

My project focuses on Refresh / Content Opportunity Scoring. The goal is to rank content pages that may need editorial review.

In the warehouse's daily performance table, one row represents one content item for one client on one reporting date. I will use March 2026 as the initial development window and aggregate daily records to one row per client and content item for scoring.

The warehouse covers approximately January 2025 to June 2026. June 2026 is reserved as a future evaluation period and will not be used to develop the label or features.

Client histories differ, so I will verify actual coverage and availability before deciding which rows are eligible.

In [1]:
print("Colab is working!")

Colab is working!


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 1: Verify the warehouse table and March 2026 partition
from huggingface_hub import list_repo_files

files = list(list_repo_files(
    "FlyRank/internship-warehouse",
    repo_type="dataset",
    token=token
))

march_files = [
    f for f in files
    if "fact_content_daily_performance" in f
    and "2026-03" in f
    and "_sample" not in f
]

print("March 2026 partition files:", len(march_files))
for f in march_files[:10]:
    print(f)

if not march_files:
    print("No March partition found; inspect the dataset layout.")


March 2026 partition files: 1
fact_content_daily_performance/month=2026-03/data_0.parquet


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Field classification — Refresh / Content Opportunity Scoring**

**Features (maximum five):** I plan to evaluate historical search impressions, historical clicks, historical average search position, content age, and historical engagement rate. Each feature must be available before the scoring date. Engagement will only be used when GA4 data is available.

**Label / proxy:** The ideal label is whether refreshing a page leads to a meaningful improvement. Since verified post-refresh outcomes are not available, I will investigate a future performance-decline indicator as an experimental proxy for review priority. This proxy does not establish whether refreshing the page would help.

**Context:** Client ID, content ID, and reporting date identify records, support grouping and time-based validation, and are not model inputs.

**Excluded:** Future clicks, future impressions, future trend measurements, and any fields calculated from the target period are excluded because they would leak information from after the scoring decision. Client identifiers, private content details, and June 2026 sample data are also excluded from model development.

**Output:** A ranked list of content pages that may deserve editorial review, with priority scores and supporting evidence.

In [5]:

# Inspect the March warehouse schema without loading all rows
!pip -q install duckdb

import duckdb
from huggingface_hub import hf_hub_url

march_url = hf_hub_url(
    repo_id="FlyRank/internship-warehouse",
    filename=march_files[0],
    repo_type="dataset"
)

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(
    "CREATE OR REPLACE SECRET hf_secret "
    "(TYPE HTTP, EXTRA_HTTP_HEADERS MAP {'Authorization': 'Bearer " + token + "'})"
)

schema = con.execute(
    "DESCRIBE SELECT * FROM read_parquet(?)",
    [march_url]
).df()

display(schema[["column_name", "column_type"]])


,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


In [6]:
print(schema["column_name"].tolist())


['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [8]:

# Section 3 — Three verification SQL queries

queries = {
    "1. Check duplicate daily records": """
        SELECT client_hash_id, content_hash_id,
               report_date, COUNT(*) AS n
        FROM read_parquet(?)
        GROUP BY client_hash_id, content_hash_id, report_date
        HAVING COUNT(*) > 1
        LIMIT 5
    """,

    "2. Check March row count and dates": """
        SELECT COUNT(*) AS total_rows,
               MIN(report_date) AS first_date,
               MAX(report_date) AS last_date
        FROM read_parquet(?)
    """,

    "3. Check GA4 availability": """
        SELECT COUNT(*) AS total_rows,
               COUNT(*) FILTER (
                   WHERE ga4_data_available IS TRUE
               ) AS ga4_available_rows
        FROM read_parquet(?)
    """
}

for name, sql in queries.items():
    print("\n" + name)
    display(con.execute(sql, [march_url]).df())



1. Check duplicate daily records


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,n



2. Check March row count and dates


,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31



3. Check GA4 availability


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows
0,9841378,413966


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Data limitations**

The FlyRank warehouse provides historical search and analytics data, but it has important limitations.

1. **Unequal history:** Clients have different starting dates for search and analytics data. Pages cannot be compared fairly without checking their available history.

2. **Missing GA4 coverage:** Some records have `ga4_data_available = FALSE`. Their zero-filled engagement values must not be interpreted as actual zero engagement.

3. **Time-window leakage:** Features must come from information available before the scoring date. Future performance measurements cannot be used as inputs.

4. **No verified refresh outcomes:** The dataset does not establish whether refreshing a particular page improves its performance. A decline-based proxy is not the same as an editor-confirmed refresh opportunity.

5. **Final-month restriction:** June 2026 is reserved for future evaluation and must not be used for developing features or labels.

**Conclusion:** The resulting scores can support editorial prioritization, but they cannot prove that refreshing a page will increase traffic.

In [9]:

# Section 4 — Record verified data limitations

print("March 2026 data-contract summary")
print("Rows: 9,841,378")
print("Observed date range: 2026-03-01 to 2026-03-31")
print("Daily-grain duplicate check: 0 duplicate groups found")
print("GA4 availability: see Section 3 query output")

print("\nLimitations:")
print("- Client history lengths differ.")
print("- GA4 zero-filled rows are not genuine zero engagement.")
print("- Future outcomes must not be used as features.")
print("- No verified post-refresh success labels are available.")
print("- June 2026 is reserved for future evaluation.")


March 2026 data-contract summary
Rows: 9,841,378
Observed date range: 2026-03-01 to 2026-03-31
Daily-grain duplicate check: 0 duplicate groups found
GA4 availability: see Section 3 query output

Limitations:
- Client history lengths differ.
- GA4 zero-filled rows are not genuine zero engagement.
- Future outcomes must not be used as features.
- No verified post-refresh success labels are available.
- June 2026 is reserved for future evaluation.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.